In [ ]:
import glob
import os
import re
import numpy as np
import scipy.io as sio
import plotly.graph_objects as go
from ipywidgets import interact, widgets
from scipy.signal import butter, filtfilt, resample

# Helper function for natural/numerical sorting
def natural_keys(text):
    return [int(c) if c.isdigit() else c for c in re.split(r'(\d+)', text)]

# 1. Get all your WFDB records and sort them numerically
record_paths = [os.path.splitext(f)[0] for f in glob.glob("/srv/home/jhyl/Afib_recurrence/diplomka/_BCOSified/finetune_run/train_data/*.hea")]
record_paths.sort(key=natural_keys)

patient_options = [(os.path.basename(p), p) for p in record_paths]

lead_options = [
    ('Lead 1 (I)', 0), ('Lead 2 (II)', 1), ('Lead 3 (III)', 2),
    ('Lead 4 (aVR)', 3), ('Lead 5 (aVL)', 4), ('Lead 6 (aVF)', 5),
    ('Lead 7 (V1)', 6), ('Lead 8 (V2)', 7), ('Lead 9 (V3)', 8),
    ('Lead 10 (V4)', 9), ('Lead 11 (V5)', 10), ('Lead 12 (V6)', 11)
]

def read_custom_header(hea_path):
    with open(hea_path, 'r') as f:
        lines = f.readlines()
    first_line_parts = lines[0].strip().split()
    fs = int(first_line_parts[2]) 
    
    sig_names = []
    for line in lines[1:]:
        line = line.strip()
        if line.startswith('#') or not line:
            continue
        parts = line.split()
        if len(parts) >= 2:
            sig_names.append(parts[1])
            
    return fs, sig_names

def preprocess_ecg_lead(signal, orig_fs, target_fs=500):
    """Applies NaN handling, resampling, bandpass filtering, and Z-scoring."""
    
    # 1. Handling NaNs
    if np.isnan(signal).any():
        if np.all(np.isnan(signal)):
            # If the entire signal is NaN, return zeros to avoid crashing
            return np.zeros(len(signal)), target_fs
        # Replace NaNs with the mean of the non-NaN values
        mean_val = np.nanmean(signal)
        signal = np.nan_to_num(signal, nan=mean_val)
        
    # 2. Resampling to 500 Hz
    if orig_fs != target_fs:
        num_samples = int(len(signal) * target_fs / orig_fs)
        signal = resample(signal, num_samples)
        current_fs = target_fs
    else:
        current_fs = orig_fs
        
    # 3. Bandpass Filtering (1 - 47 Hz)
    # Using a 4th-order Butterworth filter
    nyquist = 0.5 * current_fs
    low = 1.0 / nyquist
    high = 47.0 / nyquist
    b, a = butter(4, [low, high], btype='band')
    signal = filtfilt(b, a, signal)
    
    # 4. Z-score Standardization
    std_val = np.std(signal)
    if std_val > 0:
        signal = (signal - np.mean(signal)) / std_val
    else:
        # Prevent division by zero if the signal is perfectly flat
        signal = signal - np.mean(signal)
        
    return signal, current_fs

def plot_ecg(record_path, lead_idx):
    try:
        # Parse custom header
        orig_fs, sig_names = read_custom_header(record_path + '.hea')
        
        # Read the raw signal from the .mat file
        mat_data = sio.loadmat(record_path + '.mat')
        raw_key = 'val' if 'val' in mat_data else [k for k in mat_data.keys() if not k.startswith('__')][0]
        signals = mat_data[raw_key]
        
        if signals.shape[0] == len(sig_names):
            signals = signals.T
            
        if lead_idx >= signals.shape[1]:
            raise ValueError(f"Selected lead index {lead_idx} is out of bounds. Record only has {signals.shape[1]} leads.")
            
        # Extract just the lead we want to view
        raw_lead_signal = signals[:, lead_idx]
        lead_name = sig_names[lead_idx] if lead_idx < len(sig_names) else f"Lead {lead_idx + 1}"
        
        # Apply the Preprocessing Pipeline
        clean_signal, new_fs = preprocess_ecg_lead(raw_lead_signal.astype(float), orig_fs, target_fs=500)
        
        # Create a time axis based on the new 500Hz sampling rate
        time = np.arange(len(clean_signal)) / new_fs
        
        # Plot using Plotly
        fig = go.Figure()
        
        fig.add_trace(go.Scatter(
            x=time, 
            y=clean_signal, 
            mode='lines', 
            name=lead_name,
            line=dict(color='blue')
        ))
        
        fig.update_layout(
            title=f"Record: {os.path.basename(record_path)} | Lead: {lead_name} | Fs: {new_fs}Hz (Processed)",
            xaxis_title="Time (s)",
            yaxis_title="Amplitude (Z-score)",
            template="plotly_white",
            height=500
        )
        fig.show()
        
    except Exception as e:
        fig = go.Figure()
        fig.add_annotation(
            text=f"Error loading {os.path.basename(record_path)}:<br>{e}",
            xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False,
            font=dict(size=14, color="red")
        )
        fig.update_layout(title=f"Record: {os.path.basename(record_path)} - ERROR")
        fig.show()

# 2. Create the interactive dropdowns
patient_dropdown = widgets.Dropdown(options=patient_options, description='Patient:')
lead_dropdown = widgets.Dropdown(options=lead_options, description='Lead:')

# Link the dropdowns to the plotting function
interact(plot_ecg, record_path=patient_dropdown, lead_idx=lead_dropdown);

interactive(children=(Dropdown(description='Patient:', options=(('0', '/srv/home/jhyl/Afib_recurrence/diplomka…

In [ ]:
import glob
import os
import scipy.io as sio
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, widgets
import re

# Helper function for natural/numerical sorting
def natural_keys(text):
    """
    Splits the string into text and numbers, converting numbers to integers.
    This ensures '2' comes before '10'.
    """
    return [int(c) if c.isdigit() else c for c in re.split(r'(\d+)', text)]

# 1. Get all your WFDB records and sort them numerically
record_paths = [os.path.splitext(f)[0] for f in glob.glob("/srv/home/jhyl/Afib_recurrence/diplomka/_BCOSified/finetune_run/train_data/*.hea")]
record_paths.sort(key=natural_keys)

# Create options for the Dropdowns. 
# Format: [("Display Name", "Actual_Value_Passed_To_Function"), ...]
patient_options = [(os.path.basename(p), p) for p in record_paths]

# Create 12 standard lead options passing the column index (0 to 11)
lead_options = [
    ('Lead 1 (I)', 0), ('Lead 2 (II)', 1), ('Lead 3 (III)', 2),
    ('Lead 4 (aVR)', 3), ('Lead 5 (aVL)', 4), ('Lead 6 (aVF)', 5),
    ('Lead 7 (V1)', 6), ('Lead 8 (V2)', 7), ('Lead 9 (V3)', 8),
    ('Lead 10 (V4)', 9), ('Lead 11 (V5)', 10), ('Lead 12 (V6)', 11)
]

def read_custom_header(hea_path):
    """Custom parser for stripped-down .hea files."""
    with open(hea_path, 'r') as f:
        lines = f.readlines()
    
    first_line_parts = lines[0].strip().split()
    fs = int(first_line_parts[2]) 
    
    sig_names = []
    for line in lines[1:]:
        line = line.strip()
        if line.startswith('#') or not line:
            continue
        parts = line.split()
        if len(parts) >= 2:
            sig_names.append(parts[1])
            
    return fs, sig_names

# Updated function to accept the actual path and the lead index
def plot_ecg(record_path, lead_idx):
    try:
        # 1. Parse your custom header
        fs, sig_names = read_custom_header(record_path + '.hea')
        
        # 2. Read the raw signal from the .mat file
        mat_data = sio.loadmat(record_path + '.mat')
        raw_key = 'val' if 'val' in mat_data else [k for k in mat_data.keys() if not k.startswith('__')][0]
        signals = mat_data[raw_key]
        
        if signals.shape[0] == len(sig_names):
            signals = signals.T
            
        # Catch if a user selects a lead index that doesn't exist in the file
        if lead_idx >= signals.shape[1]:
            raise ValueError(f"Selected lead index {lead_idx} is out of bounds. Record only has {signals.shape[1]} leads.")
            
        time = np.arange(signals.shape[0]) / fs
        
        # Get actual lead name from header if available
        lead_name = sig_names[lead_idx] if lead_idx < len(sig_names) else f"Lead {lead_idx + 1}"
        
        # 3. Plot using Plotly
        fig = go.Figure()
        
        fig.add_trace(go.Scatter(
            x=time, 
            y=signals[:, lead_idx], 
            mode='lines', 
            name=lead_name
        ))
        
        fig.update_layout(
            title=f"Record: {os.path.basename(record_path)} | Lead: {lead_name} | Fs: {fs}Hz",
            xaxis_title="Time (s)",
            yaxis_title="Amplitude",
            template="plotly_white",
            height=500
        )
        fig.show()
        
    except Exception as e:
        fig = go.Figure()
        fig.add_annotation(
            text=f"Error loading {os.path.basename(record_path)}:<br>{e}",
            xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False,
            font=dict(size=14, color="red")
        )
        fig.update_layout(title=f"Record: {os.path.basename(record_path)} - ERROR")
        fig.show()

# 2. Create the interactive dropdowns
patient_dropdown = widgets.Dropdown(options=patient_options, description='Patient:')
lead_dropdown = widgets.Dropdown(options=lead_options, description='Lead:')

# Link the dropdowns to the plotting function
interact(plot_ecg, record_path=patient_dropdown, lead_idx=lead_dropdown);

interactive(children=(Dropdown(description='Patient:', options=(('0', '/srv/home/jhyl/Afib_recurrence/diplomka…